# MusicSage Separator — PyTorch (Demucs v2 style)

Разделение музыки на стемы (drums / bass / other / vocals) с помощью
1D waveform U-Net в стиле Demucs v2.

Пайплайн:
1. Загрузка датасета MUSDB18 через библиотеку `musdb` (`musdb.DB`, `track.audio`,
   `track.stems`) — без ручного вызова ffmpeg, сразу стерео float32;
2. Аугментация Demucs-стиля на волновой форме: случайный гейн, гладкий
   FIR-эквалайзер на стемы, mixup стемов между треками (chunk swap);
3. WaveUNet: энкодер stride-4 + GLU, dilated bottleneck, декодеры со skip,
   глобальный residual микса — предсказывает 4 стема напрямую в waveform-домене
   (стерео: вход (B, 2, N), выход (B, 4, 2, N));
4. Loss: L1 на волновой форме + мультирезолюционная спектральная L1
   + SI-SDR;
5. Обучение (AMP на CUDA) + инференс с кросфейдом чанков + SI-SDR-метрика.

Вдохновлено: Demucs (facebookresearch/demucs) и TF-версией `model.py` из этого репозитория.


## Установка зависимостей

```bash
pip install torch musdb soundfile numpy
```

> `musdb` тянет за собой `stempeg`, который читает `.stem.mp4` — нужен только
> `ffmpeg` в `$PATH`.

---


In [ ]:
import math
import os
import random
from collections import OrderedDict

import musdb
import numpy as np
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)
random.seed(0)


In [ ]:
# ============================================================
# CONFIG
# ============================================================

SAMPLE_RATE = 44100          # MUSDB18 native rate
CHUNK_SEC = 4
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_SEC

# WaveUNet: stride-4 свёртки, LEVELS уровней => вход паддится до кратного 4**LEVELS
STRIDE = 4
LEVELS = 5
PAD_LEN = ((CHUNK_SAMPLES + STRIDE ** LEVELS - 1) // STRIDE ** LEVELS) * STRIDE ** LEVELS

EPS = 1e-4

# Корень MUSDB18: здесь лежат папки train/ и test/ с *.stem.mp4
DB_ROOT = "musdb18/wav"

N_SOURCES = 4
SOURCE_NAMES = ["drums", "bass", "other", "vocals"]

# Мультирезолюционный STFT — только для спектрального члена loss
MR_LOSS_CONFIGS = [
    (4096, 2048),
    (2048, 1024),
    (1024, 512),
]

# Спектральная ветка гибрида: STFT микса -> log-магнитуда -> маски
SPEC_NFFT = 4096
SPEC_HOP = 1024

# Параметры WaveUNet (Demucs v2: channels=64, levels=5 -> ~47M params)
WAVE_CHANNELS = 64
WAVE_LEVELS = 5

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
# AMP (autocast + GradScaler) сильно ускоряет обучение на T4/V100;
# на MPS/CPU не поддерживается — там просто без AMP.
USE_AMP = DEVICE == "cuda"

print("device:", DEVICE, "| amp:", USE_AMP)
print("chunk:", CHUNK_SAMPLES, "samples | padded:", PAD_LEN)


## 1. Подготовка данных

MUSDB18 хранится в `musdb18/wav/{train,test}/*.stem.mp4`. Библиотека `musdb`
сама находит треки и читает стемы:

- `musdb.DB(root=DB_ROOT, subsets="train")` — база треков (по ней можно итерировать);
- `track.name`, `track.rate`, `track.duration` — метаданные;
- `track.audio` — микс `(N, 2)` стерео float32; `track.stems` — все 5 потоков `(5, N, 2)`.

**Главное ограничение скорости — декодирование.** Полный декод одной песни через
ffmpeg/stempeg занимает ~1–3 c, а random-seek по AAC почти столько же (ffmpeg
декодирует от ближайшего ключевого кадра). Поэтому датасет работает блоками:
`chunks_per_song` подряд идущих чанков берутся из одного трека (LRU-кэш на
3 песни), порядок песен перемешивается внутри датасета на каждой эпохе,
а `DataLoader` создаётся с `shuffle=False`. Итог: одна песня декодируется один
раз за эпоху (~100 декодов вместо ~800), и эпоха перестаёт упираться в ffmpeg.

Стерео сохраняем как есть: микс `(N, 2)`, стемы `(4, N, 2)`.


In [ ]:
def _stems_dict(track):
    """Все 5 стемов трека -> {mix, drums, bass, other, vocals} стерео."""
    stems = track.stems                       # (5, N, 2)
    return {
        "mix": stems[0].astype(np.float32),
        "drums": stems[1].astype(np.float32),
        "bass": stems[2].astype(np.float32),
        "other": stems[3].astype(np.float32),
        "vocals": stems[4].astype(np.float32),
    }


def load_song(track):
    """Полная песня: mix + 4 стема как стерео float32 (через API musdb)."""
    return _stems_dict(track)


In [ ]:
class SongCache:
    """LRU-кэш декодированных песен (стерео): полный декод стоит ~2 c,
    повторять его для каждого чанка нельзя."""

    def __init__(self, max_songs=3):
        self.max_songs = max_songs
        self.data = OrderedDict()

    def get(self, track):
        if track.name not in self.data:
            self.data[track.name] = load_song(track)
            while len(self.data) > self.max_songs:
                self.data.popitem(last=False)
        self.data.move_to_end(track.name)
        return self.data[track.name]


class MusicDataset(torch.utils.data.Dataset):
    """Блочная выборка по песням.

    Каждые `chunks_per_song` подряд идущих сэмплов берутся из одного трека,
    поэтому LRU-кэш попадает и каждая песня декодируется один раз за эпоху.
    Порядок песен перемешивается при старте каждой эпохи (block 0).
    DataLoader нужно создавать с shuffle=False.
    """

    def __init__(self, tracks, chunks_per_song=8, cache=None):
        self.tracks = list(tracks)
        self.chunks_per_song = chunks_per_song
        self.cache = cache if cache is not None else SongCache()
        self.order = list(range(len(self.tracks)))

    def __len__(self):
        return len(self.tracks) * self.chunks_per_song

    def __getitem__(self, idx):
        block = idx // self.chunks_per_song
        if block == 0:
            random.shuffle(self.order)   # новая эпоха: новый порядок песен
        track = self.tracks[self.order[block % len(self.tracks)]]
        song = self.cache.get(track)

        start = random.randint(0, max(0, len(song["mix"]) - CHUNK_SAMPLES))
        stop = start + CHUNK_SAMPLES

        stems = torch.as_tensor(
            np.stack([song[k] for k in SOURCE_NAMES])[:, start:stop]
        )                                   # (4, N, 2)
        mix = torch.as_tensor(song["mix"][start:stop])   # (N, 2)
        if mix.numel() < CHUNK_SAMPLES * 2:
            pad = CHUNK_SAMPLES - mix.shape[0]
            mix = F.pad(mix, (0, 0, 0, pad))
            stems = F.pad(stems, (0, 0, 0, pad))
        return mix, stems   # (N, 2) float32, (4, N, 2) float32


# musdb сам находит *.stem.mp4 в {DB_ROOT}/{train,test} и читает их через stempeg
mus_train = musdb.DB(root=DB_ROOT, subsets="train")
mus_test = musdb.DB(root=DB_ROOT, subsets="test")
print("train tracks:", len(mus_train), "| test tracks:", len(mus_test))

train_ds = MusicDataset(mus_train, chunks_per_song=8)
val_ds = MusicDataset(mus_test, chunks_per_song=2)
print("train:", len(train_ds), "| val:", len(val_ds))


## 2. Аугментация

Demucs-стайл на волновой форме, до пересборки микса:

- случайный гейн на каждый стем;
- гладкий FIR-эквалайзер на каждый стем;
- mixup (chunk swap): элементы батча с вероятностью 0.5 обмениваются
  случайными стемами с другим треком батча — модель учится разделять
  «недосмешанные» комбинации стемов;
- итоговый гейн микса.

Микс всегда пересобирается из аугментированных стемов, поэтому таргеты
согласованы с входом.


In [ ]:
def chunk_swap(stems):
    """Mixup-аугментация Demucs-стиля: каждый элемент батча с вероятностью 0.5
    обменивается случайным подмножеством стемов с другим треком батча
    (плавный вес), микс после этого пересобирается.
    Учит модель разделять даже «недосмешанные» комбинации стемов.
    Работает с (B, C, 2, N)."""
    B, C, _, _ = stems.shape
    if B < 2:
        return stems
    for i in range(B):
        if random.random() < 0.5:
            j = random.randrange(B - 1)
            j = j if j < i else j + 1      # любой другой элемент батча
            subset = [s for s in range(C) if random.random() < 0.5]
            if subset:
                g = random.uniform(0.2, 0.8)   # вес стемов из трека j
                stems[i, subset] = g * stems[j, subset] + (1 - g) * stems[i, subset]
    return stems


def augment_batch(mix, stems):
    """Demucs-стайл: случайный гейн + гладкий FIR-эквалайзер на каждый стем
    (по каналам независимо), mixup стемов между треками батча, затем микс
    пересобирается. Внутренне работает с (B, C, 2, N).
    mix: (B, N, 2), stems: (B, C, N, 2) — на выходе те же формы."""
    stems = stems.permute(0, 1, 3, 2).contiguous()        # (B, C, 2, N)
    B, C, Ch, N = stems.shape
    gains = torch.exp(torch.rand(B, C, 1, 1, device=stems.device) * 0.72 - 0.36)
    stems = stems * gains
    kern = torch.randn(B * C * Ch, 1, 16, device=stems.device)
    kern = torch.cumsum(kern, dim=2)
    kern = kern / (kern.abs().sum(dim=2, keepdim=True) + EPS)
    stems = F.conv1d(stems.view(1, B * C * Ch, N), kern, padding=8,
                     groups=B * C * Ch)
    stems = stems.squeeze(0)[:, :N].view(B, C, Ch, N)

    stems = chunk_swap(stems)              # mixup между треками батча
    mix = stems.sum(dim=1, keepdim=True)   # (B, 1, Ch, N)
    g = torch.rand(B, 1, 1, 1, device=stems.device) * 0.45 + 0.8
    mix, stems = mix * g, stems * g
    return mix.squeeze(1).permute(0, 2, 1), stems.permute(0, 1, 3, 2)


## 3. Модель — гибрид 1D waveform U-Net + 2D спектральные маски

- **Wave-ветка (1D)**: conv1d kernel=8, stride=4 + GLU; каждый уровень в 4 раза короче
  по времени, каналы удваиваются (64 → 1024);
- **Bottleneck**: два dilated-conv слоя (dilation 1 и 2) с GLU и residual-связью;
- **Декодер**: ConvTranspose stride=4 + GLU + skip-связь + conv 3x3;
- **Глобальный residual**: к выходу добавляется входной микс с обучаемым
  масштабом `dummies` — `out += (1 + dummies) * mix`: модель «вычитает»
  остальные стемы из микса, а не рисует их с нуля.

Стерео: вход `(B, 2, N)` (два канала микса), выход `(B, 4, 2, N)`
(4 стема × 2 канала). Спектральная ветка считает STFT/маски по каждому
каналу независимо с общими весами. Работает сразу с waveform — никакого
STFT на входе и на выходе.


In [ ]:
_WINDOWS = {}


def get_window(n_fft, device):
    key = (n_fft, str(device))
    if key not in _WINDOWS:
        _WINDOWS[key] = torch.sqrt(
            torch.hann_window(n_fft, periodic=True, device=device)
        )
    return _WINDOWS[key]


def stft(x, fl, fs):
    """Комплексный STFT: (..., N) -> (..., F, T). Нужен только для loss."""
    shape = x.shape[:-1]
    s = torch.stft(
        x.reshape(-1, x.shape[-1]),
        n_fft=fl, hop_length=fs, win_length=fl,
        window=get_window(fl, x.device), return_complex=True,
    )
    return s.reshape(shape + s.shape[-2:])


def log_magnitude(s):
    return torch.log1p(s.abs())


In [ ]:
_WINDOWS = {}


def get_window(n_fft, device):
    key = (n_fft, str(device))
    if key not in _WINDOWS:
        _WINDOWS[key] = torch.sqrt(
            torch.hann_window(n_fft, periodic=True, device=device)
        )
    return _WINDOWS[key]


def stft(x, fl, fs):
    """Комплексный STFT: (..., N) -> (..., F, T). Нужен только для loss."""
    shape = x.shape[:-1]
    s = torch.stft(
        x.reshape(-1, x.shape[-1]),
        n_fft=fl, hop_length=fs, win_length=fl,
        window=get_window(fl, x.device), return_complex=True,
    )
    return s.reshape(shape + s.shape[-2:])


def log_magnitude(s):
    return torch.log1p(s.abs())


class EncBlock(nn.Module):
    """Энкодер-блок: conv1d stride-4 + GLU (каналы x2 до GLU)."""
    def __init__(self, cin, cout, kernel=8, stride=4):
        super().__init__()
        self.conv = nn.Conv1d(cin, cout * 2, kernel, stride=stride,
                             padding=kernel // 2)
        self.glu = nn.GLU(dim=1)

    def forward(self, x):
        return self.glu(self.conv(x))


class Bottleneck(nn.Module):
    """Два dilated-conv слоя с GLU и residual-связью."""
    def __init__(self, channels, kernel=3):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels * 2, kernel, padding=1)
        self.glu1 = nn.GLU(dim=1)
        self.conv2 = nn.Conv1d(channels, channels * 2, kernel, padding=2, dilation=2)
        self.glu2 = nn.GLU(dim=1)

    def forward(self, x):
        return x + self.glu2(self.conv2(self.glu1(self.conv1(x))))


class DecBlock(nn.Module):
    """Декодер-блок: ConvTranspose stride-4 + GLU + skip + conv 3x3.

    Skip подрезается/допадывается до длины апсемпла, чтобы уровни
    совпадали по времени при любой длине входа.
    """
    def __init__(self, cin, cout, kernel=8, stride=4):
        super().__init__()
        self.deconv = nn.ConvTranspose1d(cin, cout * 2, kernel, stride=stride,
                                         padding=kernel // 2, output_padding=1)
        self.glu = nn.GLU(dim=1)
        self.conv = nn.Conv1d(cout, cout, 3, padding=1)

    def forward(self, x, skip=None):
        x = self.glu(self.deconv(x))
        if skip is not None:
            if x.shape[-1] > skip.shape[-1]:
                x = x[..., :skip.shape[-1]]
            elif x.shape[-1] < skip.shape[-1]:
                x = F.pad(x, (0, skip.shape[-1] - x.shape[-1]))
            x = x + skip
        return self.conv(x)


class WaveUNet(nn.Module):
    """1D waveform U-Net в стиле Demucs v2 (стерео).

    Вход: (B, 2, N) нормализованный микс; выход: (B, 4, 2, N) стемы
    (та же длина, что и вход). Стерео — это 2 канала входа, последний
    слой выдаёт n_sources * 2 каналов (пара на каждый стем).
    """

    def __init__(self, n_sources=N_SOURCES, channels=64, levels=5,
                 kernel=8, stride=4):
        super().__init__()
        self.stride = stride
        self.encoders = nn.ModuleList()
        self.decoders = nn.ModuleList()
        cin = 2
        for i in range(levels):
            cout = channels * (2 ** i)
            self.encoders.append(EncBlock(cin, cout, kernel, stride))
            cin = cout
        # декодеры: выход каналов = 2**max(0, i-1) * channels (под skip уровней),
        # верхний (i=levels-1) принимает bottleneck; нижний (i=0) — апсемпл без skip
        for i in range(levels):
            cout = channels * (2 ** max(0, i - 1))
            cin = channels * (2 ** i) if i == levels - 1 else channels * (2 ** max(0, i))
            self.decoders.append(DecBlock(cin, cout, kernel, stride))
        self.bottleneck = Bottleneck(channels * (2 ** (levels - 1)))
        self.head = nn.Conv1d(channels, n_sources * 2, 1)
        self.dummies = nn.Parameter(torch.zeros(n_sources))  # глобальный residual

    def forward(self, x):
        mix0 = x
        skips = []
        for enc in self.encoders:
            x = enc(x)
            skips.append(x)
        x = self.bottleneck(x)
        # скипы уровней: dec4 <- s3, ..., dec1 <- s0; нижний dec0 без skip
        for dec, skip in zip(self.decoders[::-1], skips[-2::-1]):
            x = dec(x, skip)
        x = self.decoders[0](x)
        if x.shape[-1] < mix0.shape[-1]:   # робастность к неделимым длинам
            x = F.pad(x, (0, mix0.shape[-1] - x.shape[-1]))
        x = self.head(x)[..., :mix0.shape[-1]]           # (B, S*2, N)
        x = x.view(x.shape[0], -1, 2, x.shape[-1])       # (B, S, 2, N)
        return x + (1 + self.dummies)[None, :, None, None] * mix0[:, None]


class SpecBlock(nn.Module):
    """2D conv-блок: conv3x3 -> GroupNorm -> SiLU -> conv3x3 -> GroupNorm + residual."""
    def __init__(self, cin, cout):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.gn1 = nn.GroupNorm(min(cout, 8), cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.gn2 = nn.GroupNorm(min(cout, 8), cout)
        self.act = nn.SiLU(inplace=True)
        self.shortcut = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x):
        h = self.act(self.gn1(self.conv1(x)))
        h = self.gn2(self.conv2(h))
        return self.act(h + self.shortcut(x))


class STFTBranch(nn.Module):
    """2D U-Net по log-магнитуде STFT микса -> 4 мягкие маски (sigmoid).

    Вход: (B, 1, F, T); выход: (B, 4, F, T) маски в [0, 1].
    Частота и время даунсемплются в 2 раза на уровень.
    """
    def __init__(self, n_sources=N_SOURCES, base=16, levels=4):
        super().__init__()
        self.enc = nn.ModuleList()
        self.dec = nn.ModuleList()
        for i in range(levels):
            cin = 1 if i == 0 else base * 2 ** (i - 1)
            cout = base * 2 ** i
            self.enc.append(SpecBlock(cin, cout))
        c = base * 2 ** (levels - 1)
        self.bottleneck = SpecBlock(c, c)
        for i in range(levels - 1, -1, -1):
            cout = base * 2 ** i
            cin = c + c if i == levels - 1 else base * 2 ** i * 3
            self.dec.append(SpecBlock(cin, cout))
        self.head = nn.Conv2d(base, n_sources, 1)

    def forward(self, spec):
        skips = []
        x = spec
        for enc in self.enc:
            x = enc(x)
            skips.append(x)
            x = F.avg_pool2d(x, 2)
        x = self.bottleneck(x)
        for dec, skip in zip(self.dec, skips[::-1]):
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear",
                              align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
        return torch.sigmoid(self.head(x))


class HybridUNet(nn.Module):
    """Гибрид: 1D waveform U-Net + 2D спектральные маски, объединение гейтом.

    Вход: (B, 2, N) нормализованный микс; выход: (B, 4, 2, N) стемы.

    - wave-ветка предсказывает стемы напрямую в waveform-домене;
    - spec-ветка строит мягкие маски по log-магнитуде STFT микса
      (канал микса обрабатывается независимо с общими весами), стемы
      восстанавливаются маскированием комплексного STFT + ISTFT;
    - выход = g * wave + (1 - g) * spec, где g — обучаемый per-stem гейт
      (init 0.5/0.5, модель сама учится, кому верить).
    """
    def __init__(self, n_sources=N_SOURCES, channels=64, levels=5,
                 nfft=4096, hop=1024, spec_base=16, spec_levels=4):
        super().__init__()
        self.wave = WaveUNet(n_sources=n_sources, channels=channels, levels=levels)
        self.spec = STFTBranch(n_sources=n_sources, base=spec_base, levels=spec_levels)
        self.nfft = nfft
        self.hop = hop
        self.gate = nn.Parameter(torch.zeros(n_sources))

    def forward(self, x):
        wav = self.wave(x)                             # (B, 4, 2, N)
        xf = x.float()
        win = torch.hann_window(self.nfft, device=x.device)
        st = torch.stft(xf.reshape(-1, x.shape[-1]), self.nfft, self.hop,
                        window=win, return_complex=True)   # (B*2, F, T)
        masks = self.spec(st.abs().log1p().unsqueeze(1))   # (B*2, 4, F, T)
        B = x.shape[0]
        spec_stems = torch.istft(
            (masks * st.unsqueeze(1)).view(B * 2 * N_SOURCES, *st.shape[-2:]),
            self.nfft, self.hop, window=win, length=x.shape[-1],
        )
        spec_stems = spec_stems.view(B, 2, N_SOURCES, x.shape[-1]) \
                                 .permute(0, 2, 1, 3)       # (B, 4, 2, N)
        g = torch.sigmoid(self.gate)[None, :, None, None]
        return (g * wav + (1 - g) * spec_stems).to(wav.dtype)


USE_HYBRID = True   # True: гибрид 1D+2D; False: чистая WaveUNet (для сравнения)
if USE_HYBRID:
    model = HybridUNet(n_sources=N_SOURCES, channels=WAVE_CHANNELS,
                       levels=WAVE_LEVELS, nfft=SPEC_NFFT, hop=SPEC_HOP).to(DEVICE)
else:
    model = WaveUNet(n_sources=N_SOURCES, channels=WAVE_CHANNELS, levels=WAVE_LEVELS).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"params: {n_params:,}")

# --- проверка прохода через сеть ---
with torch.no_grad():
    x0 = torch.randn(1, 2, PAD_LEN, device=DEVICE)
    out = model(x0)
print("input :", tuple(x0.shape))
print("output:", tuple(out.shape))


## 4. Loss

- **L1 на волновой форме** — основной член;
- **мультирезолюционная L1 на log-магнитудах STFT** — спектральная поддержка,
  дёшево заменяет гибридную спектральную ветку Demucs v3;
- **SI-SDR на волновой форме** — напрямую оптимизирует итоговую метрику.


In [ ]:
def si_sdr(est, ref, eps=1e-8):
    """SI-SDR по батчу: (B, C, 2, N) -> (B, C, 2) дБ (по каналам отдельно).
    Тензорный вариант метрики из ячейки оценки — используется как loss."""
    est = est - est.mean(-1, keepdim=True)
    ref = ref - ref.mean(-1, keepdim=True)
    alpha = (ref * est).sum(-1, keepdim=True) / (
        (ref * ref).sum(-1, keepdim=True).clamp_min(eps)
    )
    dist = est - alpha * ref
    return 10 * torch.log10(
        (alpha ** 2 * (ref * ref).sum(-1, keepdim=True) + eps)
        / ((dist * dist).sum(-1, keepdim=True) + eps)
    )


class WaveLoss(nn.Module):
    """L1 waveform + мультирезолюционная спектральная L1 + SI-SDR."""

    def __init__(self, alpha=1.0, beta=0.5, delta=0.2,
                 spec_configs=MR_LOSS_CONFIGS):
        super().__init__()
        self.alpha, self.beta, self.delta = alpha, beta, delta
        self.spec_configs = spec_configs

    def forward(self, est, tgt):
        # est, tgt: (B, C, 2, N) — нормализованные waveform стемы (стерео)
        l_wave = F.l1_loss(est, tgt)

        l_spec = 0.0
        for fl, fs in self.spec_configs:
            e = log_magnitude(stft(est, fl, fs))
            t = log_magnitude(stft(tgt, fl, fs))
            l_spec += F.l1_loss(e, t)
        l_spec /= len(self.spec_configs)

        l_sisdr = -si_sdr(est, tgt).mean()

        return self.alpha * l_wave + self.beta * l_spec + self.delta * l_sisdr


loss_fn = WaveLoss().to(DEVICE)
print(loss_fn)


## 5. Обучение

- **нормализация чанка**: микс и стемы делятся на std микса (per-item,
  по каналам отдельно), чтобы L1 работал в одном масштабе для треков
  любой громкости;
- **AMP** (autocast + GradScaler) на CUDA — ~x2 быстрее на T4;
- grad clipping, cosine annealing, best-чекпойнт по val loss.

Ориентиры: на T4 batch=8, 4s-чанки, ~6-7 мин/эпоху. Стерео удваивает
объём данных и память на батч — при OOM уменьшить batch_size до 4
(эпоха станет вдвое длиннее).


In [ ]:
def run_epoch(model, loader, optimizer, loss_fn, device, train=True, scaler=None):
    model.train(train)
    total, n = 0.0, 0
    with torch.enable_grad() if train else torch.no_grad():
        for mix, stems in loader:
            mix = mix.to(device, non_blocking=True)        # (B, N, 2)
            stems = stems.to(device, non_blocking=True)    # (B, 4, N, 2)
            if train:
                mix, stems = augment_batch(mix, stems)
            # per-chunk нормализация по std микса (по каналам отдельно)
            scale = mix.std(dim=1).clamp_min(1e-4)         # (B, 2)
            mix = F.pad((mix / scale[:, None, :]).permute(0, 2, 1),
                        (0, PAD_LEN - CHUNK_SAMPLES))      # (B, 2, N+pad)
            stems = stems.permute(0, 1, 3, 2) / scale[:, None, :, None]   # (B, 4, 2, N)

            with torch.autocast(device_type=device, enabled=scaler is not None):
                pred = model(mix)[..., :CHUNK_SAMPLES]
                loss = loss_fn(pred, stems)

            if train:
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
                    optimizer.step()
            total += float(loss)
            n += 1
    return total / n


EPOCHS = 40
# shuffle=False: перемешивание песен делает сам датасет блоками (см. выше),
# чтобы кэш декодирования попадал; num_workers>0 — декод в фоновых процессах.
train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True
)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True
)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
os.makedirs("checkpoints_pytorch", exist_ok=True)

best_val = float("inf")
for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(model, train_loader, optimizer, loss_fn, DEVICE,
                   train=True, scaler=scaler)
    va = run_epoch(model, val_loader, optimizer, loss_fn, DEVICE, train=False)
    scheduler.step()
    is_best = va < best_val
    if is_best:
        best_val = va
    torch.save(model.state_dict(), f"checkpoints_pytorch/sep_{epoch:02d}.pt")
    if is_best:
        torch.save(model.state_dict(), "checkpoints_pytorch/sep_best.pt")
    print(f"epoch {epoch:2d} | train {tr:.4f} | val {va:.4f}"
          + ("  (best)" if is_best else ""))


## 6. Инференс

Песню режем на перекрывающиеся чанки (50% overlap, кросфейд по краям),
предсказываем стемы сразу в waveform-домене (без STFT), нормализацию
и её обратное преобразование делаем так же, как в обучении.


In [ ]:
def separate_track(model, mix, device, source_idx=0):
    """Полный трек -> стерео стем (np.float32, (N, 2)),
    перекрывающиеся чанки с кросфейдом."""
    model.eval()
    n = len(mix)
    hop = CHUNK_SAMPLES // 2
    ramp = int(0.25 * SAMPLE_RATE)

    out = np.zeros((n, 2), dtype=np.float32)
    weights = np.zeros(n, dtype=np.float32)
    ramp_w = np.linspace(0, 1, ramp, dtype=np.float32)

    with torch.no_grad():
        for st in range(0, max(1, n - CHUNK_SAMPLES + 1), hop):
            chunk = mix[st:st + CHUNK_SAMPLES]            # (N', 2)
            if len(chunk) < CHUNK_SAMPLES:
                chunk = np.pad(chunk, ((0, CHUNK_SAMPLES - len(chunk)), (0, 0)))
            x = torch.from_numpy(chunk).to(device)        # (N, 2)
            scale = x.std(dim=0).clamp_min(1e-4)          # (2,)
            xp = F.pad((x / scale).t().unsqueeze(0),
                       (0, PAD_LEN - CHUNK_SAMPLES))      # (1, 2, N+pad)
            est = model(xp)[0, source_idx, :, :CHUNK_SAMPLES] * scale[:, None]
            est = est.t().cpu().numpy()                   # (N, 2)

            w = np.ones(CHUNK_SAMPLES, dtype=np.float32)
            w[:ramp] = ramp_w
            w[-ramp:] = ramp_w[::-1]

            stop = min(st + CHUNK_SAMPLES, n)
            out[st:stop] += est[:stop - st] * w[:stop - st, None]
            weights[st:stop] += w[:stop - st]

    ok = weights > 0
    out[ok] /= weights[ok, None]
    return out


def separate_file(model, track, device, source_idx=3):
    """Стем из musdb.Track (0 drums, 1 bass, 2 other, 3 vocals)."""
    song = load_song(track)
    stem = separate_track(model, song["mix"], device, source_idx=source_idx)
    return stem, song[SOURCE_NAMES[source_idx]], song["mix"]


In [ ]:
# Демо: отделить вокал из первого трека теста
test_track = mus_test[0]

est_vocals, ref_vocals, mix = separate_file(model, test_track, DEVICE, source_idx=3)

os.makedirs("output", exist_ok=True)
sf.write("output/mix.wav", mix, SAMPLE_RATE)
sf.write("output/vocals_ref.wav", ref_vocals, SAMPLE_RATE)
sf.write("output/vocals_pred.wav", est_vocals, SAMPLE_RATE)

print("wrote output/*.wav |", test_track.name)


## 8. Разделение своей песни

Любой аудиофайл (mp3/wav/flac/...) — MUSDB18 не нужен, модель обучена в этой
сессии. В Colab: загрузи файл через sidebar (панель файлов) или `files.upload()`
и укажи имя в `user_path`.


In [ ]:
# ============================================================
# Разделение своей песни
# ============================================================

import stempeg


def load_audio(path):
    """Любой аудиофайл (wav/mp3/flac/...) -> стерео float32 (N, 2), 44100 Гц.
    Моно-файлы дублируются в оба канала."""
    stems, _ = stempeg.read_stems(path, sample_rate=SAMPLE_RATE,
                                  ffmpeg_format="s16le")
    audio = np.squeeze(stems)               # (N,) или (N, C)
    if audio.ndim == 1:
        audio = np.stack([audio, audio], axis=1)
    elif audio.shape[1] != 2:
        audio = audio.mean(axis=1)
        audio = np.stack([audio, audio], axis=1)
    return audio.astype(np.float32)


# --- настройки ---
user_path = "my_song.mp3"                   # <-- путь к своему файлу
# Вместо in-memory модели можно загрузить чекпойнт:
# model.load_state_dict(torch.load("checkpoints_pytorch/sep_best.pt",
#                                  map_location=DEVICE))

if not os.path.exists(user_path):
    raise FileNotFoundError(f"файл не найден: {user_path}")

mix = load_audio(user_path)
print(f"{user_path}: {len(mix) / SAMPLE_RATE:.1f} s @ {SAMPLE_RATE} Hz")

os.makedirs("output", exist_ok=True)
for idx, name in enumerate(SOURCE_NAMES):
    est = separate_track(model, mix, DEVICE, source_idx=idx)
    sf.write(f"output/{name}.wav", est, SAMPLE_RATE)
    print(f"saved output/{name}.wav")

sf.write("output/mix.wav", mix, SAMPLE_RATE)
print("done")


## 7. Метрика — SI-SDR

Оценка качества отделённого стема на тестовых треках (SDR в дБ, чем больше — тем лучше).


In [ ]:
def si_sdr(estimate, reference):
    """SI-SDR в дБ: (N,) или (N, 2) -> скаляр (среднее по каналам)."""
    eps = 1e-8
    if estimate.ndim == 1:
        estimate = estimate[:, None]
        reference = reference[:, None]
    estimate = estimate - estimate.mean(axis=0, keepdims=True)
    reference = reference - reference.mean(axis=0, keepdims=True)
    alpha = np.sum(reference * estimate, axis=0) / (
        np.sum(reference * reference, axis=0) + eps
    )
    distortion = estimate - alpha * reference
    sdr = 10 * np.log10(
        alpha ** 2 * np.sum(reference ** 2, axis=0)
        / (np.sum(distortion ** 2, axis=0) + eps)
    )
    return float(sdr.mean())


def evaluate(model, tracks, max_songs=3, source_idx=3):
    sdr_list = []
    for track in tracks[:max_songs]:
        est, ref, _ = separate_file(model, track, DEVICE, source_idx=source_idx)
        n = min(len(est), len(ref))
        sdr_list.append(si_sdr(est[:n], ref[:n]))
        print(f"{track.name[:40]:42s} SI-SDR = {sdr_list[-1]:.1f} dB")
    return np.mean(sdr_list), sdr_list


mean_sdr, per_song = evaluate(model, mus_test, max_songs=3, source_idx=3)
print(f"\nmean SI-SDR (vocals): {mean_sdr:.1f} dB")
